# Tier 3a — Criterion-Level LLM Reranker

**Architecture:** clf-v4 → top-50 → criterion-level Qwen2.5-7B rerank

For each (patient, trial) pair the LLM scores every parsed inclusion and exclusion
criterion separately, then aggregates into a single trial score:
```
trial_score = agg(inc_scores) − agg(exc_scores)
```
Three aggregations are tested: `mean`, `sum`, `mean_inc_max_exc`.

**Requires:** `criteria_data.jsonl` on Drive (produced by `dataprep_criteria.ipynb`).

**Tier 2a baseline (TREC22, clf→Qwen flat-doc top-50):** NDCG@10 = 0.6485

| Stage | TREC22 NDCG@10 |
|---|---|
| clf-v4 only | 0.6388 |
| clf→Qwen flat-doc top-50 (Tier 2a) | 0.6485 |
| clf→Qwen criterion-level top-50 (Tier 3a) | _this notebook_ |

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(result.stdout.strip() or 'No GPU detected')

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm
!pip install -q sympy==1.13.1

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # paste your READ token here (huggingface.co/settings/tokens)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT      = '/content/drive/MyDrive/ct_data23'
QRELS_PATH     = f'{DATA_ROOT}/unified_qrels.jsonl'
CRITERIA_PATH  = f'{DATA_ROOT}/criteria_data.jsonl'    # from dataprep_criteria.ipynb
TIER2A_PATH    = f'{DATA_ROOT}/llm_reranker_results.json'

CLF_CHECKPOINT = 'semaj83/ctmatch-clf-v4'
LLM_MODEL      = 'Qwen/Qwen2.5-7B-Instruct'

# Gates — must reproduce before trusting Tier 3a numbers
CLF_GATE_ALL   = 0.7460   # NDCG@10, all 184 topics
CLF_GATE_T22   = 0.6388   # NDCG@10, TREC22 only (clean holdout)
GATE_TOL       = 0.005

TOP_K_RERANK   = 50
CRIT_BATCH     = 32        # criterion prompts per LLM forward pass
TREC22_ONLY    = True      # evaluate on TREC22 only (clean holdout)

In [ ]:
import json

topic2text   = {}
topic2source = {}
topic2rel    = {}   # topic_id -> {doc_id: label}

with open(QRELS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        tid = rec['topic_id']
        if TREC22_ONLY and rec['source'] != 'trec22':
            continue
        topic2text[tid] = rec['topic_text']
        topic2source[tid] = rec['source']
        if tid not in topic2rel:
            topic2rel[tid] = {}
        topic2rel[tid][rec['doc_id']] = rec['label']

print(f'Topics: {len(topic2text)}')
print(f'Sources: {set(topic2source.values())}')
avg_pool = sum(len(d) for d in topic2rel.values()) / len(topic2rel)
print(f'Avg judged docs/topic: {avg_pool:.1f}')

In [ ]:
from datasets import load_dataset

index2docid_ds  = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
doc_texts_ds    = load_dataset('semaj83/ctmatch_ir', data_files='doc_texts.txt',   split='train')

index2docid = [row['text'].strip() for row in index2docid_ds]
docid2index = {nct_id: idx for idx, nct_id in enumerate(index2docid)}
docid2text  = {nct_id: doc_texts_ds[idx]['text'] for idx, nct_id in enumerate(index2docid)}

print(f'Corpus size: {len(index2docid):,} trials')

# Check judged pool coverage
all_judged = {nct_id for d in topic2rel.values() for nct_id in d}
missing = all_judged - set(docid2index)
print(f'Judged trials in corpus: {len(all_judged) - len(missing)} / {len(all_judged)}  (missing: {len(missing)})')

In [ ]:
nctid2crit = {}
with open(CRITERIA_PATH) as f:
    for line in f:
        rec = json.loads(line)
        nctid2crit[rec['nct_id']] = {
            'include_criteria': rec['include_criteria'],
            'exclude_criteria': rec['exclude_criteria'],
        }

covered = sum(
    1 for nct_id in all_judged
    if nct_id in nctid2crit and
       (nctid2crit[nct_id]['include_criteria'] or nctid2crit[nct_id]['exclude_criteria'])
)
print(f'Criteria data loaded: {len(nctid2crit):,} trials')
print(f'Judged trials with ≥1 criterion: {covered} / {len(all_judged)} ({100*covered/len(all_judged):.1f}%)')

# Distribution
judged_recs = [nctid2crit[n] for n in all_judged if n in nctid2crit]
inc_lens = [len(r['include_criteria']) for r in judged_recs]
exc_lens = [len(r['exclude_criteria']) for r in judged_recs]
print(f'Inc criteria/trial — mean: {sum(inc_lens)/len(inc_lens):.1f}  max: {max(inc_lens)}')
print(f'Exc criteria/trial — mean: {sum(exc_lens)/len(exc_lens):.1f}  max: {max(exc_lens)}')

In [ ]:
import math

def calc_ndcg(ranked_ids, doc2rel, k=10):
    dcg = sum(
        doc2rel.get(doc_id, 0) / math.log2(rank + 1)
        for rank, doc_id in enumerate(ranked_ids[:k], start=1)
    )
    ideal = sorted(doc2rel.values(), reverse=True)[:k]
    idcg  = sum(rel / math.log2(rank + 1) for rank, rel in enumerate(ideal, start=1))
    return dcg / idcg if idcg > 0 else 0.0

def calc_mrr(ranked_ids, doc2rel):
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc2rel.get(doc_id, 0) >= 2:
            return 1.0 / rank
    return 0.0

def eval_ranking(topic2ranked, topic2rel, k=10):
    ndcgs, mrrs = [], []
    for topic_id, ranked_ids in topic2ranked.items():
        doc2rel = topic2rel.get(topic_id, {})
        ndcgs.append(calc_ndcg(ranked_ids, doc2rel, k))
        mrrs.append(calc_mrr(ranked_ids, doc2rel))
    return {
        f'ndcg@{k}': sum(ndcgs) / len(ndcgs) if ndcgs else 0.0,
        'mrr':       sum(mrrs)  / len(mrrs)  if mrrs  else 0.0,
        'n_topics':  len(ndcgs),
    }

## Stage 1 — clf-v4 scoring

Score all judged docs per topic with the cross-encoder, build the gate-validated top-50 per topic.
Delete clf-v4 before loading the LLM.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

clf_tokenizer = AutoTokenizer.from_pretrained(CLF_CHECKPOINT)
clf_model     = AutoModelForSequenceClassification.from_pretrained(CLF_CHECKPOINT)
clf_model.eval()
clf_model.cuda()

id2label = clf_model.config.id2label
relevant_col = [int(k) for k, v in id2label.items() if v == 'relevant'][0]
print(f'Label map: {id2label}')
print(f'relevant_col: {relevant_col}')

In [ ]:
from tqdm.auto import tqdm

def clf_batch_score(topic_text, doc_texts, batch_size=64):
    scores = []
    for i in range(0, len(doc_texts), batch_size):
        batch  = doc_texts[i:i+batch_size]
        pairs  = [(topic_text, dt) for dt in batch]
        enc    = clf_tokenizer(pairs, padding=True, truncation=True,
                               max_length=512, return_tensors='pt').to(clf_model.device)
        with torch.no_grad():
            logits = clf_model(**enc).logits
        probs = F.softmax(logits, dim=1)[:, relevant_col]
        scores.extend(probs.cpu().tolist())
    return scores

clf_topic2ranked = {}  # topic_id -> [(nct_id, score), ...] sorted descending
for topic_id in tqdm(topic2text, desc='clf scoring'):
    doc2rel  = topic2rel[topic_id]
    nct_ids  = [nid for nid in doc2rel if nid in docid2text]
    texts    = [docid2text[nid] for nid in nct_ids]
    scores   = clf_batch_score(topic2text[topic_id], texts)
    ranked   = sorted(zip(nct_ids, scores), key=lambda x: x[1], reverse=True)
    clf_topic2ranked[topic_id] = ranked

In [ ]:
clf_topic2ids = {tid: [nct_id for nct_id, _ in ranked] for tid, ranked in clf_topic2ranked.items()}
clf_res       = eval_ranking(clf_topic2ids, topic2rel)
clf_ndcg      = clf_res['ndcg@10']
gate          = CLF_GATE_T22 if TREC22_ONLY else CLF_GATE_ALL
status        = '✓ PASS' if clf_ndcg >= gate - GATE_TOL else '✗ FAIL'
print(f'{status}  clf-v4  NDCG@10={clf_ndcg:.4f}  MRR={clf_res["mrr"]:.4f}  (gate={gate})')
if '✗' in status:
    raise RuntimeError('clf-v4 gate failed — check CLF_CHECKPOINT and TREC22_ONLY settings')

In [ ]:
import gc
del clf_model, clf_tokenizer
gc.collect()
torch.cuda.empty_cache()
print('clf-v4 freed')

## Stage 2 — criterion-level LLM scoring

For each trial in clf-v4's top-50, score every inclusion and exclusion criterion separately.
Aggregate criterion scores into a single trial score and re-sort.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer  = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model  = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map='auto')
llm_model.eval()
llm_device = next(llm_model.parameters()).device
print(f'LLM on: {llm_device}')

if tokenizer.padding_side != 'left':
    tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def get_answer_ids(word):
    """Return all single-token IDs for common surface forms of a yes/no answer word."""
    forms = [word, word.capitalize(), word.upper(), ' ' + word, ' ' + word.capitalize()]
    ids = set()
    for form in forms:
        toks = tokenizer.encode(form, add_special_tokens=False)
        if len(toks) == 1:
            ids.add(toks[0])
    return list(ids)

yes_ids = get_answer_ids('yes')
no_ids  = get_answer_ids('no')
assert yes_ids, 'No single-token yes IDs found'
assert no_ids,  'No single-token no IDs found'
print(f'yes token IDs: {yes_ids} → {[tokenizer.decode([i]) for i in yes_ids]}')
print(f'no  token IDs: {no_ids}  → {[tokenizer.decode([i]) for i in no_ids]}')

In [ ]:
def make_inc_prompt(topic_text: str, criterion: str) -> str:
    msgs = [{'role': 'user', 'content': (
        f'Patient: {topic_text}\n\n'
        f'Inclusion criterion: {criterion}\n\n'
        'Does the patient satisfy this inclusion criterion? Answer yes or no.'
    )}]
    return tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

def make_exc_prompt(topic_text: str, criterion: str) -> str:
    msgs = [{'role': 'user', 'content': (
        f'Patient: {topic_text}\n\n'
        f'Exclusion criterion: {criterion}\n\n'
        'Does this exclusion criterion apply to the patient? Answer yes or no.'
    )}]
    return tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

In [ ]:
# Spot-check: show one inc and one exc prompt for the first topic
sample_topic = list(topic2text.keys())[0]
sample_nct   = list(topic2rel[sample_topic].keys())[0]
sample_crit  = nctid2crit.get(sample_nct, {'include_criteria': ['[no data]'], 'exclude_criteria': ['[no data]']})

print('=== INC PROMPT ===')
print(make_inc_prompt(topic2text[sample_topic], sample_crit['include_criteria'][0] if sample_crit['include_criteria'] else '[empty]')[:600])
print('\n=== EXC PROMPT ===')
print(make_exc_prompt(topic2text[sample_topic], sample_crit['exclude_criteria'][0] if sample_crit['exclude_criteria'] else '[empty]')[:600])

In [ ]:
def logprob_score_batch(prompts: list[str], max_length: int = 1024) -> torch.Tensor:
    """
    Returns log P(yes) − log P(no) for each prompt.
    Left-pads the batch; reads logits at the final position (next-token prediction).
    """
    enc = tokenizer(
        prompts, padding=True, truncation=True,
        max_length=max_length, return_tensors='pt'
    ).to(llm_device)

    with torch.no_grad():
        logits = llm_model(**enc).logits   # (B, seq, vocab)

    next_tok_logits = logits[:, -1, :]     # (B, vocab) — next-token position
    log_probs       = torch.log_softmax(next_tok_logits, dim=-1)

    yes_score = torch.logsumexp(log_probs[:, yes_ids], dim=1)   # list indexing, device-safe
    no_score  = torch.logsumexp(log_probs[:, no_ids],  dim=1)
    return (yes_score - no_score).cpu()

In [ ]:
# Quick sanity: scores should be higher for clearly-met criteria
topic_text = topic2text[sample_topic]
test_prompts = [
    make_inc_prompt(topic_text, 'Patient must have a cancer diagnosis'),
    make_inc_prompt(topic_text, 'Patient must be pregnant'),
    make_exc_prompt(topic_text, 'Patient must not have any prior cancer history'),
]
test_scores = logprob_score_batch(test_prompts)
labels = ['inc: cancer diagnosis', 'inc: pregnant', 'exc: no prior cancer']
for label, score in zip(labels, test_scores.tolist()):
    print(f'  {score:+.3f}  {label}')

In [ ]:
from collections import defaultdict

# trial_score = agg(inc_scores) − agg(exc_scores)
#   inc_score > 0 → patient meets criterion (good)
#   exc_score > 0 → criterion excludes patient (bad)
#
# mean:             normalises by #criteria, fair across trial sizes
# sum:              rewards more evidence of eligibility (longer criterion lists)
# mean_inc_max_exc: strict exclusion model — any hard exclusion dominates

AGG_FNS = {
    'mean':             lambda inc, exc: (
        (sum(inc)/len(inc) if inc else 0.0) - (sum(exc)/len(exc) if exc else 0.0)
    ),
    'sum':              lambda inc, exc: (
        (sum(inc) if inc else 0.0) - (sum(exc) if exc else 0.0)
    ),
    'mean_inc_max_exc': lambda inc, exc: (
        (sum(inc)/len(inc) if inc else 0.0) - (max(exc) if exc else 0.0)
    ),
}


def score_criteria_for_topic(topic_text, nct_ids, nctid2crit, batch_size=CRIT_BATCH):
    """
    Score all (nct_id, criterion) pairs for a topic.
    Returns {nct_id: {agg_name: score}} for each aggregation in AGG_FNS.
    """
    prompts, meta = [], []
    for nct_id in nct_ids:
        crit = nctid2crit.get(nct_id, {'include_criteria': [], 'exclude_criteria': []})
        for ct in crit['include_criteria']:
            prompts.append(make_inc_prompt(topic_text, ct))
            meta.append((nct_id, True))
        for ct in crit['exclude_criteria']:
            prompts.append(make_exc_prompt(topic_text, ct))
            meta.append((nct_id, False))

    raw = []
    for i in range(0, len(prompts), batch_size):
        raw.extend(logprob_score_batch(prompts[i:i+batch_size]).tolist())

    inc_scores = defaultdict(list)
    exc_scores = defaultdict(list)
    for (nct_id, is_inc), score in zip(meta, raw):
        (inc_scores if is_inc else exc_scores)[nct_id].append(score)

    return {
        nct_id: {agg: fn(inc_scores.get(nct_id, []), exc_scores.get(nct_id, []))
                 for agg, fn in AGG_FNS.items()}
        for nct_id in nct_ids
    }

In [ ]:
criteria_topic2ranked = {agg: {} for agg in AGG_FNS}
n_crit_passes = 0

for topic_id in tqdm(topic2text, desc='Criterion scoring'):
    clf_ranked  = clf_topic2ranked[topic_id]
    top_k_ncts  = [nct_id for nct_id, _ in clf_ranked[:TOP_K_RERANK]]
    rest_ncts   = [nct_id for nct_id, _ in clf_ranked[TOP_K_RERANK:]]

    trial_scores = score_criteria_for_topic(topic2text[topic_id], top_k_ncts, nctid2crit)

    n_crit_passes += sum(
        len(nctid2crit.get(n, {}).get('include_criteria', [])) +
        len(nctid2crit.get(n, {}).get('exclude_criteria', []))
        for n in top_k_ncts
    )

    for agg in AGG_FNS:
        reranked = sorted(top_k_ncts, key=lambda n: trial_scores[n][agg], reverse=True)
        criteria_topic2ranked[agg][topic_id] = reranked + rest_ncts

print(f'Total criterion-level forward passes: {n_crit_passes:,}')

In [ ]:
print('=== Criterion-level reranker results (TREC22) ===')
print(f'  clf-v4 baseline:  NDCG@10={clf_ndcg:.4f}')
print()

agg_results = {}
for agg, topic2ranked in criteria_topic2ranked.items():
    res = eval_ranking(topic2ranked, topic2rel)
    agg_results[agg] = res
    delta = res['ndcg@10'] - clf_ndcg
    sign  = '+' if delta >= 0 else ''
    print(f'  {agg:20s}  NDCG@10={res["ndcg@10"]:.4f}  MRR={res["mrr"]:.4f}  Δ={sign}{delta:.4f}')

In [ ]:
# Compare to Tier 2a (flat-doc reranker)
if os.path.exists(TIER2A_PATH):
    with open(TIER2A_PATH) as f:
        tier2a = json.load(f)
    t2a_ndcg = tier2a.get('clf_llm_top50', 'N/A')
    print(f'Tier 2a (flat-doc, TREC22): NDCG@10={t2a_ndcg}')
    for agg, res in agg_results.items():
        if isinstance(t2a_ndcg, float):
            delta = res['ndcg@10'] - t2a_ndcg
            sign  = '+' if delta >= 0 else ''
            print(f'  vs Tier 2a  {agg:20s}  Δ={sign}{delta:.4f}')
else:
    print(f'Tier 2a results not found at {TIER2A_PATH} — run rerank_llm.ipynb first')

In [ ]:
best_agg = max(agg_results, key=lambda a: agg_results[a]['ndcg@10'])
best_res = agg_results[best_agg]

output = {
    'clf_v4':              clf_ndcg,
    'agg_results':         agg_results,
    'best_agg':            best_agg,
    'best_ndcg@10':        best_res['ndcg@10'],
    'best_mrr':            best_res['mrr'],
    'top_k_rerank':        TOP_K_RERANK,
    'crit_forward_passes': n_crit_passes,
    'trec22_only':         TREC22_ONLY,
    'llm_model':           LLM_MODEL,
    'clf_checkpoint':      CLF_CHECKPOINT,
}

save_path = f'{DATA_ROOT}/criteria_reranker_results.json'
with open(save_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Results → {save_path}')
print(f'Best aggregation: {best_agg}')
print(f'  NDCG@10={best_res["ndcg@10"]:.4f}  MRR={best_res["mrr"]:.4f}')